# Calculate general catchment statistics

In [ ]:
# Imports
import geopandas as gpd

In [ ]:
# General settings

# Catchment characterisation shape files location
path_catchment_characterisation = '../1_catchment_shape_files'

# System characterisation shape files location
path_system_characterisation = '../2_system_shape_files'

## Catchment area

In [ ]:
# Load geodataframe from shape file
shp_path_catchment = f'{path_catchment_characterisation}/catchment_outline/Study_area.shp'
gdf_catchment = gpd.read_file(
        shp_path_catchment, columns=['NAME', 'geometry'])
gdf_catchment = gdf_catchment.to_crs(epsg=32610)

# Calculate area
area_catchment_m2 = gdf_catchment.area.sum()
print(f'Catchment area: {round(area_catchment_m2 / 1000000, 2)} km2')

## Catchment elevation

In [ ]:
# Load geodataframe from shape file
shp_path_2d_zones_elevation = f'{path_catchment_characterisation}/elevation/2D Zones elevation.shp'
gdf_2d_zones = gpd.read_file(shp_path_2d_zones_elevation)

# Calculate mesh element areas
gdf_2d_zones['area_m2'] = gdf_2d_zones.to_crs('EPSG:32610').area

In [ ]:
# Find min and max elevations
elevation_min = gdf_2d_zones['elevation2'].min()
elevation_max = gdf_2d_zones['elevation2'].max()
print(f'Min elevation = {round(elevation_min, 2)} m')
print(f'Max elevation = {round(elevation_max, 2)} m')

In [ ]:
# Calculate area below 0m
area_below_0_m2 = gdf_2d_zones[gdf_2d_zones['elevation2'] < 0]['area_m2'].sum()
print(f'{round(area_below_0_m2 / 1000000, 2)} km2 below 0 mAOD')

In [ ]:
# Calculate mean elevation
elevation_mean = (gdf_2d_zones['elevation2'] * gdf_2d_zones['area_m2']).sum() / gdf_2d_zones['area_m2'].sum()
print(f'Mean elevation = {round(elevation_mean, 2)} m')

In [ ]:
# Calculate elevation percentiles
gdf_2d_zones.sort_values('elevation2', inplace=True)     # Sort by elevation
gdf_2d_zones['cumulative_area'] = gdf_2d_zones['area_m2'].cumsum() # Calculate cumulative area

area_m2_total = gdf_2d_zones['area_m2'].sum()
for quantile in [10, 25, 50, 75, 80, 90, 95]:
    elevation_quantile = gdf_2d_zones.loc[
        gdf_2d_zones['cumulative_area'] >= quantile / 100 * area_m2_total, 'elevation2'
    ].iloc[0]
    print(f'{quantile} percentile elevation = {round(elevation_quantile, 2)} m')

## Watercourse length

In [ ]:
# Load 'links' geodataframe from shape file and extract fluvial / overland system
shp_path_links = f'{path_system_characterisation}/links/Links.shp'
gdf_links = gpd.read_file(shp_path_links)

gdf_fluvial = gdf_links[
    (gdf_links['system_typ'].isin(['other', 'overland'])) |
    (gdf_links['link_type'] == 'River reach')
]

# Reproject geodataframe to a metric CRS (BNG)
gdf_fluvial = gdf_fluvial.to_crs(epsg=27700)

In [ ]:
# Calculate total length
fluvial_length_km = gdf_fluvial.length.sum() / 1000
print(f'Total length of fluvial system links = {round(fluvial_length_km, 1)} km')